# Project Setup

This notebook downloads and prepares every resource needed across the whole pipeline: the Romantic source texts (Project Gutenberg), the pretrained BERT checkpoint used later for NER fine-tuning, and the spaCy and NLTK/WordNet resources.

**Outputs:**
- `data/corpus/*.txt` and `data/corpus/corpus_metadata.csv` - extracted source texts + metadata
- A locally cached `bert-base-cased` checkpoint - ready for the NER fine-tuning notebook
- Locally cached `en_core_web_sm` and `en_core_web_trf` spaCy models
- Locally cached NLTK WordNet data

Run all cells top to bottom. Re-running is safe - downloads/caches are skipped if already present.

---

In [1]:
import os
import re
import csv
import time
import requests

# 1. Source texts

## Configuration

All texts are public-domain editions extracted from Project Gutenberg.

The files often contain unpredictable front matter, including transcriber credits, editorial introductions, and multi-level tables of contents. To bypass this inconsistent formatting, the extraction parameters below use a "first line" strategy.

Each entry defines one passage: `start_marker` (required), and either `end_marker`, `end_after_marker` (cuts right after this phrase - used when the poem's own last line needs to be included, avoiding editorial footnotes that follow it), or neither (whole remaining section).
`notes` records the specific reasoning behind each passage's boundaries.

13 passages across 5 authors: Coleridge, Wordsworth (3), Percy Shelley (4), Mary Shelley (2), Byron (2).

In [2]:
NOTEBOOK_DIR = os.path.abspath(os.getcwd())

if os.path.basename(NOTEBOOK_DIR) == "notebooks":
    PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
else:
    PROJECT_ROOT = NOTEBOOK_DIR

RAW_CACHE_DIR = os.path.join(PROJECT_ROOT, "data", "raw_cache")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "data", "corpus")
METADATA_PATH = os.path.join(OUTPUT_DIR, "corpus_metadata.csv")
PREVIEW_ONLY = False

GUTENBERG_TXT_URL = "https://www.gutenberg.org/cache/epub/{id}/pg{id}.txt"

TEXTS = [
    {
        "author": "Samuel Taylor Coleridge",
        "work": "The Rime of the Ancient Mariner",
        "gutenberg_id": 151,
        "extract_full": False,
        "start_marker": "It is an ancient Mariner,",
        "end_marker": None,
        "end_after_marker": None,
        "max_words": None,
        "notes": "Whole text is the poem, but there is the title to remove.",
    },
    {
        "author": "William Wordsworth",
        "work": "Lines Written a Few Miles above Tintern Abbey",
        "gutenberg_id": 9622,
        "extract_full": False,
        "start_marker": "Five years have passed; five summers, with the length",
        "end_marker": None,
        "end_after_marker": "for thy sake.",  # poem's actual last line; cuts before the printed footnotes
        "max_words": None,
        "notes": "Final poem in the 1798 Lyrical Ballads volume. Extraction ends at the poem's last line, before the editor's footnotes.",
    },
    {
        "author": "William Wordsworth",
        "work": "Lines Written in Early Spring",
        "gutenberg_id": 9622,
        "extract_full": False,
        "start_marker": "I heard a thousand blended notes,",
        "end_marker": "THE THORN",
        "end_after_marker": None,
        "max_words": None,
        "notes": "Starts at the poem's first line; ends at the next poem's title in the same file.",
    },
    {
        "author": "William Wordsworth",
        "work": "The Tables Turned",
        "gutenberg_id": 9622,
        "extract_full": False,
        "start_marker": "Up! up! my friend, and clear your looks,",
        "end_marker": "OLD MAN TRAVELLING",
        "end_after_marker": None,
        "max_words": None,
        "notes": "Starts at the poem's first line; ends at the next poem's title.",
    },
    {
        "author": "Percy Shelley",
        "work": "Ode to the West Wind",
        "gutenberg_id": 4800,
        "extract_full": False,
        "start_marker": "O wild West Wind, thou breath of Autumn’s being,",
        "end_marker": "AN EXHORTATION",
        "end_after_marker": None,
        "max_words": None,
        "notes": "Starts at the poem's first line to bypass the dual table of contents and heading.",
    },
    {
        "author": "Percy Shelley",
        "work": "Mont Blanc",
        "gutenberg_id": 4800,
        "extract_full": False,
        "start_marker": "The everlasting universe of things",
        "end_marker": None,
        "end_after_marker": "Silence and solitude were vacancy?",  # poem's last line
        "max_words": None,
        "notes": "Starts at the poem's first line to bypass the dual table of contents.",
    },
    {
        "author": "Percy Shelley",
        "work": "The Cloud",
        "gutenberg_id": 4800,
        "extract_full": False,
        "start_marker": "I bring fresh showers for the thirsting flowers,",
        "end_marker": None,
        "end_after_marker": "I arise and unbuild it again.",  # poem's last line
        "max_words": None,
        "notes": "Starts at the poem's first line to bypass the table of contents and structural headings.",
    },
    {
        "author": "Percy Shelley",
        "work": "To a Skylark",
        "gutenberg_id": 4800,
        "extract_full": False,
        "start_marker": "Hail to thee, blithe Spirit!",
        "end_marker": None,
        "end_after_marker": "as I am listening now.",  # poem's last line
        "max_words": None,
        "notes": "Starts at the poem's first line to bypass the table of contents and structural headings.",
    },
    {
        "author": "Mary Shelley",
        "work": "Frankenstein - Letters 1-4 (Arctic frame)",
        "gutenberg_id": 84,
        "extract_full": False,
        "start_marker": "You will rejoice to hear that no disaster has accompanied the",
        "end_marker": "Chapter 1",
        "end_after_marker": None,
        "max_words": None,
        "notes": "Starts at the first line of the letter to automatically skip the table-of-contents block preceding it.",
    },
    {
        "author": "Mary Shelley",
        "work": "Frankenstein - Chapter 10 (Mont Blanc / glacier scene)",
        "gutenberg_id": 84,
        "extract_full": False,
        "start_marker": "I spent the following day roaming through the valley. I stood beside",
        "end_marker": "Chapter 11",
        "end_after_marker": None,
        "max_words": None,
        "notes": "Starts at the first line of the chapter to automatically skip the table-of-contents block.",
    },
    {
        "author": "Lord Byron",
        "work": "Childe Harold's Pilgrimage - Canto III (Lake Leman / Alps)",
        "gutenberg_id": 5131,
        "extract_full": False,
        "start_marker": "Clear, placid Leman",
        "end_marker": "CANTO THE FOURTH",
        "end_after_marker": None,
        "max_words": None,
        "notes": "Extraction begins at the Lake Leman transition (stanza LXVIII), skipping the earlier Waterloo/Rhine material, and runs to the end of the canto.",
    },
    {
        "author": "Lord Byron",
        "work": "Manfred - Act I Scene II (Jungfrau cliff)",
        "gutenberg_id": 20158,
        "extract_full": False,
        "start_marker": " _Man_. The spirits I have raised abandon me,",
        "end_marker": "ACT II",
        "end_after_marker": None,
        "max_words": None,
        "notes": "Extraction begins at the first line of dialogue for the specific Scene, bypassing the Gutenberg formatting and act headers.",
    },
    {
        "author": "Lord Byron",
        "work": "Manfred - Act II Scene II (Witch of the Alps)",
        "gutenberg_id": 20158,
        "extract_full": False,
        "start_marker": "It is not noon--the Sunbow's rays",
        "end_marker": "SCENE III",
        "end_after_marker": None,
        "max_words": None,
        "notes": "Extraction begins at the first line of dialogue for the specific Scene, bypassing the Gutenberg formatting and act headers.",
    },
]

## Extraction and Cleaning

The core functions that turn one raw Gutenberg download into one clean corpus file:

- **`fetch_gutenberg_text()`** - downloads the plain-text edition of a book, caching it in `RAW_CACHE_DIR` so re-running the notebook never re-downloads a book it already has.
- **`extract_passage()`** - slices out one specific passage from a book's full text, anchored to `start_marker`. Supports three ways to define the end: `end_after_marker` (include up through this phrase), `end_marker` (stop before this phrase), or `max_words` (word-count cap) — falls through to "rest of the text" if none are given.
- **`strip_paratext()`** - removes editorial and structural material that isn't part of the poem/prose itself.
- **`safe_filename()`** / **`word_count()`** - builds a filesystem-safe `author_work.txt` name, and counts words for logging/sanity-checking passage lengths.

In [ ]:
def fetch_gutenberg_text(gutenberg_id: int) -> str:
    os.makedirs(RAW_CACHE_DIR, exist_ok=True)
    cache_path = os.path.join(RAW_CACHE_DIR, f"{gutenberg_id}.txt")

    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8", errors="replace") as f:
            return f.read()

    url = GUTENBERG_TXT_URL.format(id=gutenberg_id)
    resp = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    text = resp.text

    with open(cache_path, "w", encoding="utf-8") as f:
        f.write(text)

    time.sleep(1)
    return text


def strip_boilerplate(text: str) -> str:
    start_pattern = re.compile(
        r"\*\*\*\s*START OF (THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*",
        re.IGNORECASE | re.DOTALL,
    )
    end_pattern = re.compile(
        r"\*\*\*\s*END OF (THE|THIS) PROJECT GUTENBERG EBOOK",
        re.IGNORECASE,
    )
    start_match = start_pattern.search(text)
    end_match = end_pattern.search(text)
    body_start = start_match.end() if start_match else 0
    body_end = end_match.start() if end_match else len(text)
    return text[body_start:body_end].strip()


def slice_by_word_count(text: str, max_words: int) -> str:
    matches = list(re.finditer(r"\S+", text))
    if len(matches) <= max_words:
        return text
    end_pos = matches[max_words - 1].end()
    return text[:end_pos]


def extract_passage(body: str, entry: dict) -> str:
    work = entry["work"]
    start_marker = entry["start_marker"]
    end_marker = entry.get("end_marker")
    end_after_marker = entry.get("end_after_marker")
    max_words = entry.get("max_words")

    start_idx = body.find(start_marker)
    if start_idx == -1:
        raise ValueError(f"[{work}] start_marker {start_marker!r} not found.")

    if end_after_marker is not None:
        marker_idx = body.find(end_after_marker, start_idx + len(start_marker))
        if marker_idx == -1:
            raise ValueError(f"[{work}] end_after_marker {end_after_marker!r} not found.")
        return body[start_idx : marker_idx + len(end_after_marker)].strip()

    if end_marker is None and max_words is None:
        return body[start_idx:].strip()

    if end_marker is None and max_words is not None:
        return slice_by_word_count(body[start_idx:], max_words).strip()

    end_idx = body.find(end_marker, start_idx + len(start_marker))
    if end_idx == -1:
        raise ValueError(f"[{work}] end_marker {end_marker!r} not found.")
        
    return body[start_idx:end_idx].strip()


def clean_text(passage: str) -> str:
    passage = re.sub(r"\r\n", "\n", passage)
    passage = re.sub(r"\n{3,}", "\n\n", passage)
    passage = re.sub(r"[ \t]+", " ", passage)
    return passage.strip()


def strip_paratext(text: str) -> str:
    
    # unclosed stage directions starting with '[_' and ending with '._' or ']' (Manfred)
    text = re.sub(r"\[_.*?(?:\._|\])\s*", "", text, flags=re.DOTALL)
    
    # general bracketed notes (e.g., [4])
    text = re.sub(r"\[.*?\]\s*", "", text, flags=re.DOTALL)
    
    # voice indications (coleridge)
    text = re.sub(r"^(FIRST|SECOND) VOICE\.?\s*", "", text, flags=re.MULTILINE)
    
    # letter sign-offs/headers (mary shelley)
    text = re.sub(r"^Letter \d+\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"^_To Mrs\. Saville, England\._\s*", "", text, flags=re.MULTILINE)
    
    # underscore line-number markers
    text = re.sub(r"_\d+\b", "", text)

    cleaned_lines = []
    for line in text.split("\n"):
        stripped = line.strip()
        
        # skip standalone Roman or Arabic numerals
        if re.fullmatch(r"([IVXLCDM]+|\d+)\.?", stripped):
            continue
            
        # skip generic part/chapter/canto/stanza headers
        if re.fullmatch(r"(PART THE|CHAPTER|CANTO|STANZA|BOOK)\s+[A-Za-z0-9]+\.?", stripped, re.IGNORECASE):
            continue
            
        cleaned_lines.append(line)
        
    text = "\n".join(cleaned_lines)
    
    # clean up excess newlines left by removed lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def safe_filename(author: str, work: str) -> str:
    name = f"{author}_{work}".lower()
    name = re.sub(r"[^a-z0-9]+", "_", name).strip("_")
    return name + ".txt"


def word_count(text: str) -> int:
    return len(text.split())

## Run

Runs the full pipeline over every entry in `TEXTS`: download (or read from cache) → strip boilerplate → extract the specific passage → strip paratext → clean formatting → save. Writes `corpus_metadata.csv` recording every text's source, word count, and output path.

In [ ]:
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    metadata_rows = []

    for entry in TEXTS:
        author = entry["author"]
        work = entry["work"]
        g_id = entry["gutenberg_id"]

        print(f"\n=== {author} -- {work} ===")
        if entry.get("notes"):
            print(f"NOTE: {entry.get('notes', '')}")

        try:
            raw = fetch_gutenberg_text(g_id)
            body = strip_boilerplate(raw)

            if entry["extract_full"]:
                passage = body
            else:
                passage = extract_passage(body, entry)

            passage = strip_paratext(passage)
            passage = clean_text(passage)
            n_words = word_count(passage)

            if PREVIEW_ONLY:
                print(f"  Extracted {n_words} words. Preview:")
                print("  ---- START ----")
                print(" ", passage[:300].replace("\n", " "))
                print("  ...")
                print(" ", passage[-300:].replace("\n", " "))
                print("  ----  END  ----")
                continue

            out_filename = safe_filename(author, work)
            out_path = os.path.join(OUTPUT_DIR, out_filename)
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(passage)

            metadata_rows.append({
                "author": author,
                "work": work,
                "gutenberg_id": g_id,
                "source_url": GUTENBERG_TXT_URL.format(id=g_id),
                "n_words": n_words,
                "output_file": out_path,
            })
            print("File correctly saved")

        except Exception as e:
            print(f"  ERROR: {e}")

    if not PREVIEW_ONLY and metadata_rows:
        metadata_dir = os.path.dirname(METADATA_PATH)
        if metadata_dir:
            os.makedirs(metadata_dir, exist_ok=True)
            with open(METADATA_PATH, "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(
                    f,
                    fieldnames=["author", "work", "gutenberg_id", "source_url",
                                "n_words", "output_file"],
                )
                writer.writeheader()
                writer.writerows(metadata_rows)
            print("\nCorpus metadata written to corpus_metadata.csv")

    if PREVIEW_ONLY:
        print("\n\nPREVIEW_ONLY is True - nothing was saved.")


main()


=== Samuel Taylor Coleridge -- The Rime of the Ancient Mariner ===
NOTE: Whole text is the poem, but there is the title to remove.
File correctly saved

=== William Wordsworth -- Lines Written a Few Miles above Tintern Abbey ===
NOTE: Final poem in the 1798 Lyrical Ballads volume. Extraction ends at the poem's last line, before the editor's footnotes.
File correctly saved

=== William Wordsworth -- Lines Written in Early Spring ===
NOTE: Starts at the poem's first line; ends at the next poem's title in the same file.
File correctly saved

=== William Wordsworth -- The Tables Turned ===
NOTE: Starts at the poem's first line; ends at the next poem's title.
File correctly saved

=== Percy Shelley -- Ode to the West Wind ===
NOTE: Starts at the poem's first line to bypass the dual table of contents and heading.
File correctly saved

=== Percy Shelley -- Mont Blanc ===
NOTE: Starts at the poem's first line to bypass the dual table of contents.
File correctly saved

=== Percy Shelley -- The

---

# 2. Cache BERT checkpoint

Downloads `bert-base-cased` (tokenizer + pretrained weights) and stores it locally, so the NER fine-tuning notebook doesn't have to wait on a download later.

`bert-base-cased` (not the uncased version) is used because capitalization is meaningful in this corpus - e.g. distinguishing personified "Nature" from generic lowercase "nature" will be one of the annotation guideline's central rules, and an uncased model would throw that distinction away before the classifier ever sees it.

In [5]:
from transformers import AutoTokenizer, AutoModel

BERT_MODEL_NAME = "bert-base-cased"
AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
AutoModel.from_pretrained(BERT_MODEL_NAME)
print(f"{BERT_MODEL_NAME} cached locally.")

bert-base-cased cached locally.


---

# 3. Cache spaCy models

Downloads and caches the spaCy models used later in the pipeline: `en_core_web_sm` (used in annotation.ipynb) and `en_core_web_trf` (used in dependency_analysis.ipynb).

In [6]:
import spacy
import spacy.cli

SPACY_MODELS = ["en_core_web_sm", "en_core_web_trf"]

for model_name in SPACY_MODELS:
    try:
        spacy.load(model_name)
        print(f"{model_name} already cached locally.")
    except OSError:
        print(f"Downloading {model_name} ...")
        spacy.cli.download(model_name)
        print(f"{model_name} cached locally.")

en_core_web_sm already cached locally.
en_core_web_trf already cached locally.


---

# 4. Cache WordNet data

In [7]:
import nltk
nltk.download('wordnet', quiet=True)

True